# 06 — Integração WVS + V-Dem

**Objetivo:** Construir a base integrada entre perfis afetivos individuais e indicadores institucionais do V-Dem em nível país-ano.

# 06 — Integração WVS + V-Dem

## Objetivo

Este notebook tem como objetivo integrar a base individual do World Values Survey, já enriquecida com os perfis afetivos produzidos no Experimento 4, aos indicadores institucionais do V-Dem.

A unidade de análise será preservada no nível individual, incorporando variáveis contextuais de país-ano. O produto final desta etapa será a base integrada:

`dados/integrados/wvs_vdem.parquet`

Esta etapa não recalcula os fatores nem os perfis afetivos já produzidos. Ela utiliza exclusivamente arquivos persistidos em disco, em conformidade com a lógica modular e reprodutível do projeto.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Localiza automaticamente a raiz do projeto
BASE = Path.cwd().resolve().parent

dados = BASE / "dados"
brutos = dados / "brutos"
processados = dados / "processados"
integrados = dados / "integrados"
figuras = BASE / "figuras"
resultados = BASE / "resultados"

print("Projeto carregado em:")
print(BASE)

Projeto carregado em:
C:\Users\tmaci\Documents\Doutorado_Populismo


## 1. Carregamento dos dados

In [6]:
# ============================================================
# CARREGAMENTO DAS BASES
# ============================================================

wvs = pd.read_parquet(processados / "wvs_indicadores_w7.parquet")

vdem = pd.read_csv(
    brutos / "VDEM" / "V-Dem-CY-Core-v16_csv" / "V-Dem-CY-Core-v16.csv",
    low_memory=False
)

print("WVS:", wvs.shape)
print("V-Dem:", vdem.shape)

WVS: (24053, 42)
V-Dem: (28092, 1908)


In [7]:
# ============================================================
# INSPEÇÃO DAS VARIÁVEIS DE CHAVE
# ============================================================

print("=" * 60)
print("COLUNAS DO WVS")
print("=" * 60)

for c in wvs.columns:
    print(c)

print("\n")

print("=" * 60)
print("COLUNAS DO V-DEM")
print("=" * 60)

for c in vdem.columns:
    if "country" in c.lower() or "year" in c.lower():
        print(c)

COLUNAS DO WVS
national_pride
willing_to_fight
grandiosity
reject_different_race
reject_immigrants
reject_homosexuals
reject_other_religion
reject_different_language
immigration_restriction
outgroup_hostility
respect_authority
strong_leader
army_rule
religious_law_no_parties
army_as_democracy
authoritarianism
politics_important
interest_politics
discuss_politics
petition
boycott
demonstration
contact_official
encourage_action
political_mobilization
low_interpersonal_trust
distrust_first_time
distrust_other_religion
distrust_other_nationality
low_social_trust
low_human_rights_respect
no_say_government
distrust_parliament
distrust_government
distrust_parties
grievance
trust_churches
trust_armed_forces
trust_police
trust_state_authority
collective_narcissism_proxy_w7
imagined_subcommunity_proxy_w7


COLUNAS DO V-DEM
country_name
country_text_id
country_id
year


In [8]:
# ============================================================
# INSPEÇÃO DA BASE ORIGINAL DO WVS
# ============================================================

wvs_original = pd.read_parquet(processados / "wvs_tese.parquet")

print("Dimensão:", wvs_original.shape)

print("\nColunas disponíveis:\n")

for c in wvs_original.columns:
    print(c)

Dimensão: (39720, 66)

Colunas disponíveis:

S002VS
S020
S003
COUNTRY_ALPHA
S018
G006
E012
G001
G002
G024
G025
E291
E124
E128
E125
E069_07
E069_11
E069_12
A124_02
A124_05
A124_06
A124_09
A124_10
A124_12
A124_18
A124_27
A124_34
A124_43
E143
E018
E114
E116
E117
E117B
E118
E119
E121
E122
E123
E228
A004
E023
A062
A063
A068
A102
E025
E026
E027
E025B
E026B
E221B
E257
E287
E288
A165
A168
A168A
G007_34_B
G007_35_B
G007_36_B
E069_01
E069_02
E069_06
E069_17
E033


In [9]:
# ============================================================
# RECUPERAÇÃO DAS CHAVES DE INTEGRAÇÃO
# ============================================================

# Recupera apenas as chaves da base original
chaves = wvs_original[["COUNTRY_ALPHA", "S020"]].copy()

# Anexa à base analítica
wvs = pd.concat([chaves, wvs], axis=1)

# Renomeia para facilitar os merges futuros
wvs = wvs.rename(columns={
    "COUNTRY_ALPHA": "country_text_id",
    "S020": "year"
})

print(wvs.shape)

wvs.head()

(39720, 44)


,country_text_id,year,national_pride,willing_to_fight,grandiosity,reject_different_race,reject_immigrants,reject_homosexuals,reject_other_religion,reject_different_language,...,distrust_parliament,distrust_government,distrust_parties,grievance,trust_churches,trust_armed_forces,trust_police,trust_state_authority,collective_narcissism_proxy_w7,imagined_subcommunity_proxy_w7
0,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,1.000000,1.000000,0.750000,0.000000,0.000000,1.000000,0.333333,0.524769,0.413657
1,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,1.000000,0.666667,0.666667,0.650000,0.000000,0.000000,0.000000,0.000000,0.425463,0.342130
2,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,1.000000,0.666667,1.000000,0.533333,1.000000,1.000000,0.333333,0.777778,0.458102,0.504398
3,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,1.000000,1.000000,0.733333,0.666667,0.333333,1.000000,0.666667,0.508796,0.453241
4,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.666667,0.333333,0.666667,0.479167,0.333333,0.666667,0.666667,0.555556,0.326157,0.335417


In [10]:
# ============================================================
# PREPARAÇÃO DO V-DEM
# ============================================================

print("Número de variáveis:", len(vdem.columns))

print("\nPrimeiras 50 variáveis:\n")

for c in vdem.columns[:50]:
    print(c)

Número de variáveis: 1908

Primeiras 50 variáveis:

country_name
country_text_id
country_id
year
historical_date
project
historical
histname
codingstart
codingend
codingstart_contemp
codingend_contemp
codingstart_hist
codingend_hist
gapstart1
gapstart2
gapstart3
gapend1
gapend2
gapend3
gap_index
COWcode
v2x_polyarchy
v2x_polyarchy_codelow
v2x_polyarchy_codehigh
v2x_polyarchy_sd
v2x_libdem
v2x_libdem_codelow
v2x_libdem_codehigh
v2x_libdem_sd
v2x_partipdem
v2x_partipdem_codelow
v2x_partipdem_codehigh
v2x_partipdem_sd
v2x_delibdem
v2x_delibdem_codelow
v2x_delibdem_codehigh
v2x_delibdem_sd
v2x_egaldem
v2x_egaldem_codelow
v2x_egaldem_codehigh
v2x_egaldem_sd
v2x_api
v2x_api_codelow
v2x_api_codehigh
v2x_api_sd
v2x_mpi
v2x_mpi_codelow
v2x_mpi_codehigh
v2x_mpi_sd


In [11]:
# ============================================================
# SELEÇÃO DAS VARIÁVEIS DO V-DEM
# ============================================================

vdem_base = vdem[
    [
        "country_text_id",
        "year",
        "country_name",
        "country_id",
        "v2x_polyarchy",
        "v2x_libdem",
        "v2x_partipdem",
        "v2x_delibdem",
        "v2x_egaldem",
        "v2x_api",
        "v2x_mpi"
    ]
].copy()

print(vdem_base.shape)

vdem_base.head()

(28092, 11)


,country_text_id,year,country_name,country_id,v2x_polyarchy,v2x_libdem,v2x_partipdem,v2x_delibdem,v2x_egaldem,v2x_api,v2x_mpi
0,MEX,1789,Mexico,3,0.027,0.044,0.006,NaN,NaN,0.055,0.0
1,MEX,1790,Mexico,3,0.027,0.044,0.006,NaN,NaN,0.055,0.0
2,MEX,1791,Mexico,3,0.027,0.044,0.006,NaN,NaN,0.055,0.0
3,MEX,1792,Mexico,3,0.027,0.044,0.006,NaN,NaN,0.055,0.0
4,MEX,1793,Mexico,3,0.027,0.044,0.006,NaN,NaN,0.055,0.0


In [12]:
# ============================================================
# BASE INSTITUCIONAL (V-DEM)
# ============================================================

institucional = vdem_base.copy()

print("Observações:", institucional.shape[0])
print("Países:", institucional["country_text_id"].nunique())
print(
    "Período:",
    institucional["year"].min(),
    "-",
    institucional["year"].max()
)

Observações: 28092
Países: 202
Período: 1789 - 2025


In [13]:
# ============================================================
# VERIFICAÇÃO DE DUPLICIDADES
# ============================================================

duplicados = institucional.duplicated(
    subset=["country_text_id", "year"]
).sum()

print("Duplicidades:", duplicados)

Duplicidades: 0


## 2. Preparação

In [14]:
# ============================================================
# INTEGRAÇÃO WVS + V-DEM
# ============================================================

wvs_vdem = wvs.merge(
    institucional,
    on=["country_text_id", "year"],
    how="left",
    validate="many_to_one"
)

print("WVS original:", wvs.shape)
print("Base integrada:", wvs_vdem.shape)

wvs_vdem.head()

WVS original: (39720, 44)
Base integrada: (39720, 53)


,country_text_id,year,national_pride,willing_to_fight,grandiosity,reject_different_race,reject_immigrants,reject_homosexuals,reject_other_religion,reject_different_language,...,imagined_subcommunity_proxy_w7,country_name,country_id,v2x_polyarchy,v2x_libdem,v2x_partipdem,v2x_delibdem,v2x_egaldem,v2x_api,v2x_mpi
0,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.413657,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
1,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.342130,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
2,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.504398,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
3,ARG,2013,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.453241,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645
4,ARG,2013,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.335417,Argentina,37.0,0.772,0.588,0.519,0.564,0.607,0.899,0.645


In [15]:
# ============================================================
# VALIDAÇÃO DA INTEGRAÇÃO
# ============================================================

sem_vdem = wvs_vdem["country_name"].isna().sum()

print("Respondentes sem correspondência V-Dem:", sem_vdem)
print("Percentual:", round(sem_vdem / len(wvs_vdem) * 100, 2), "%")

wvs_vdem.loc[
    wvs_vdem["country_name"].isna(),
    ["country_text_id", "year"]
].drop_duplicates().sort_values(["country_text_id", "year"])

Respondentes sem correspondência V-Dem: 1127
Percentual: 2.84 %


,country_text_id,year
29576,PRI,2018


## 3. Análise

In [16]:
# ============================================================
# DIAGNÓSTICO DO CASO PRI
# ============================================================

vdem[
    vdem["country_text_id"].str.contains("PR", na=False)
][["country_name", "country_text_id"]].drop_duplicates()

,country_name,country_text_id
3192,Portugal,PRT
6423,North Korea,PRK
24557,Paraguay,PRY
26914,Parma,PRM


A integração entre o WVS e o V-Dem resultou em uma taxa de correspondência de 97,16%. Os 2,84% de observações não integradas correspondem exclusivamente a respondentes de Porto Rico (PRI), unidade amostral presente no WVS, mas não contemplada como Estado soberano na base V-Dem. Esses casos permaneceram com valores ausentes para as variáveis institucionais e foram excluídos apenas das análises que demandaram indicadores do V-Dem.

In [17]:
# ============================================================
# VALIDAÇÃO FINAL DA BASE
# ============================================================

print("Respondentes:", len(wvs_vdem))
print("Países:", wvs_vdem["country_text_id"].nunique())
print("Anos:", wvs_vdem["year"].nunique())

print("\nValores ausentes por variável institucional:\n")

print(
    wvs_vdem[
        [
            "v2x_polyarchy",
            "v2x_libdem",
            "v2x_partipdem",
            "v2x_delibdem",
            "v2x_egaldem",
            "v2x_api",
            "v2x_mpi"
        ]
    ].isna().sum()
)

Respondentes: 39720
Países: 17
Anos: 11

Valores ausentes por variável institucional:

v2x_polyarchy    1127
v2x_libdem       1127
v2x_partipdem    1127
v2x_delibdem     1127
v2x_egaldem      1127
v2x_api          1127
v2x_mpi          1127
dtype: int64


In [18]:
# ============================================================
# BASE ANALÍTICA MULTINÍVEL
# ============================================================

base_multinivel = wvs_vdem.copy()

# Salva em Parquet
base_multinivel.to_parquet(
    integrados / "base_multinivel.parquet",
    index=False
)

print("Base salva com sucesso.")
print(integrados / "base_multinivel.parquet")

Base salva com sucesso.
C:\Users\tmaci\Documents\Doutorado_Populismo\dados\integrados\base_multinivel.parquet


# 3. Análise da integração

A integração entre os indicadores individuais da World Values Survey (WVS) e os indicadores institucionais do Varieties of Democracy (V-Dem) foi realizada por meio das variáveis `country_text_id` e `year`, preservando cada respondente como unidade de análise.

A validação confirmou inexistência de duplicidades na base institucional para cada combinação país-ano, garantindo uma correspondência unívoca entre os bancos de dados. Após a integração, foram incorporados os principais índices agregados do V-Dem, permitindo que cada indivíduo passasse a carregar também informações relativas ao contexto institucional em que a entrevista foi realizada.

Observou-se taxa de correspondência superior a 97% dos respondentes. Os casos não integrados concentram-se exclusivamente em Porto Rico (PRI, 2018), território que não possui observações correspondentes na versão utilizada do V-Dem. Trata-se, portanto, de uma limitação da cobertura das bases de dados, e não de erro no procedimento de integração.

Como resultado desta etapa, foi produzida uma base analítica multinível que servirá como insumo para os experimentos subsequentes. Nenhuma transformação estatística foi realizada neste notebook além da integração, validação e persistência da base consolidada.

## Resumo da etapa

- Base individual: WVS Wave 7
- Base institucional: V-Dem v16
- Chave de integração: `country_text_id` + `year`
- Unidade de análise preservada: respondente
- Correspondência obtida: >97%
- Casos sem correspondência: Porto Rico (PRI, 2018)
- Produto gerado: `base_multinivel.parquet`

## 4. Exportação dos resultados